# Dynamic Bayesian Network — Machine-Failure Prediction from Alarm Data

> **Pure-DBN, all-alarms variant.** I train a real `DynamicBayesianNetwork` with
> `dbn.fit()` and query it with `DBNInference`, using **all 94 alarms**. A single
> `State` node cannot depend on 94 alarms individually (its table would need ~4¹⁸⁸
> entries), so I summarise every window's alarm activity into a few aggregate
> features and let those feed the state.

I build a Dynamic Bayesian Network that predicts if a machine will transition to a
**Failure** state in the next time window, given its current state and the alarm
behaviour observed in the current window.

$$P(\text{State}_{t+1} = \text{Failure} \mid \text{State}_t,\ \text{alarm features at } t)$$

### 1. What is a Bayesian Network (BN)?
A Bayesian Network is a probabilistic graphical model that represents a set of random variables and their conditional dependencies through a Directed Acyclic Graph (DAG). Grounded in Bayes' Theorem, it lets me compactly represent the joint probability distribution of an entire system by exploiting the conditional independencies between variables. Bayesian Networks are widely used for diagnostic and predictive reasoning under uncertainty.

### 2. What is a Dynamic Bayesian Network (DBN)?
A Dynamic Bayesian Network extends the traditional Bayesian Network to model temporal, sequential, or time-series data. While a standard BN captures a static "snapshot" of a system, a DBN connects multiple BNs sequentially across discrete time slices. It assumes the Markov property — the state of the system at time $t$ depends only on the state at time $t-1$ — which keeps the model computationally tractable.

### 3. How are temporal dependencies represented in a DBN?
Temporal dependencies are modelled with **transition edges** (also called inter-slice edges) that cross from one time slice to the next. For example, a directed edge connects a node at time $t$ (e.g. `Machine_State_t`) to a node at time $t+1$ (`Machine_State_t+1`). These cross-slice arrows explicitly capture how past observations and history influence the future state.

### 4. What are Nodes, Directed Edges, and Conditional Probability Tables (CPTs)?
- **Nodes:** the building blocks of the graph, each representing a random variable. In this exercise the nodes represent variables such as *Alarm Count*, *Alarm Duration*, and the *Machine State* (Running / Failure).
- **Directed Edges:** arrows connecting nodes to depict causal influence or direct conditional dependency. An arrow from node $A$ to node $B$ means that $B$ is probabilistically conditioned on $A$.
- **Conditional Probability Tables (CPTs):** tables attached to each node that quantify the probability distribution of that node given every possible combination of its parents' states.

### Import Libraries

I import all the libraries at first that are needed for the exercise:
- **pandas** and **numpy** — for data loading and manipulation
- **pgmpy** — the Bayesian Network library, I use two parts of it:
  - `DynamicBayesianNetwork` — to declare and train the DBN with `dbn.fit()`
  - `DBNInference` — for probabilistic inference on the DBN
- **warnings** — just to suppress noisy output

In [ ]:
import pandas as pd
import numpy as np
from pgmpy.models import DynamicBayesianNetwork as DBN
from pgmpy.inference import DBNInference
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## Data Preprocessing & Feature Extraction

This part covers **Steps 0 and 1** of the exercise:

1. Load the dataset and compute each alarm's **active duration** in seconds
   (`end_alarm - start_alarm`)
2. Summarise **all 94 alarms** in each window into a few **aggregate features**
   (total events, total duration, number of distinct alarms)
3. Discretise those aggregates into categories, because Bayesian Networks work with
   discrete values
4. Generate **transitions**, pairs of consecutive windows (t → t+1), which is what
   the DBN learns from.

### Load & compute duration

I parse the timestamps and derive `duration_seconds` for each alarm record.

In [ ]:
df = pd.read_csv('dataset_exercise.csv', delimiter=';')
df['start_alarm'] = pd.to_datetime(df['start_alarm'])
df['end_alarm']   = pd.to_datetime(df['end_alarm'])
df['duration_seconds'] = (df['end_alarm'] - df['start_alarm']).dt.total_seconds()

print(f"Loaded {len(df):,} rows | {df['time_window'].nunique()} time windows | "
      f"{df['alarm_id'].nunique()} unique alarm IDs")

### Aggregate all 94 alarms into per-window summaries

A `State` node can't depend on 94 alarms individually, so I summarise every window's
alarm activity into the exercise's two Step-1 attributes — **Count** and **Duration** —
computed at the window level, two ways each:

**Count**
- `total_alarms` — total alarm events that fired (across all 94 types)
- `num_active`   — how many *distinct* alarms fired

**Duration**
- `total_dur`    — the combined active duration of all alarms
- `max_dur`      — the duration of the **longest single alarm** (the strongest failure
  signal: windows with a very long alarm fail far more often)

In [ ]:
records = []
for w, g in df.groupby('time_window'):
    records.append({
        'time_window' : w,
        'State'       : g['machine_state'].mode().iloc[0],
        'total_alarms': len(g),                       # all alarm events
        'total_dur'   : g['duration_seconds'].sum(),  # combined duration
        'num_active'  : g['alarm_id'].nunique(),      # distinct alarms firing
        'max_dur'     : g['duration_seconds'].max(),  # longest single alarm
    })
df_agg = pd.DataFrame(records).sort_values('time_window').reset_index(drop=True)

print(f"{len(df_agg)} windows built")
print("State distribution:", df_agg['State'].value_counts().to_dict())
df_agg.head()

### Discretisation

I convert each aggregate into **4 categories — None / Low / Medium / High** (matching
the count/duration scheme from the exercise): a value of 0 becomes `None`, and the
**nonzero** values are split at their tertiles (33rd and 66th percentiles). Computing
the cut-points on the nonzero values keeps the zeros from skewing the bins.
Discretisation is required because Bayesian Networks work with discrete values.

In [ ]:
AGG = ['total_alarms', 'total_dur', 'num_active', 'max_dur']

def make_tertile(series):
    # zeros -> None, nonzero values split at tertiles -> 4 categories: None/Low/Medium/High
    nonzero = series[series > 0]
    lo, hi = nonzero.quantile([1/3, 2/3])
    def f(v):
        if v == 0:    return 'None'
        elif v <= lo: return 'Low'
        elif v <= hi: return 'Medium'
        else:         return 'High'
    return f

discretisers = {c: make_tertile(df_agg[c]) for c in AGG}
for c in AGG:
    df_agg[c + '_cat'] = df_agg[c].map(discretisers[c])

AGG_CATS = [c + '_cat' for c in AGG]
for c in AGG_CATS:
    print(c, df_agg[c].value_counts().to_dict())

### Generate training transitions (Step 3)

A DBN learns from pairs of consecutive windows: the situation at time t paired with
the state at time t+1. I use `shift(-1)` to pair each window with the next, then I
filter out the gaps in the `time_window` column — windows whose IDs are not exactly 1
apart are not consecutive.

In [ ]:
df_agg['State_next']  = df_agg['State'].shift(-1)
df_agg['next_window'] = df_agg['time_window'].shift(-1)
df_transitions = df_agg[
    df_agg['next_window'] == df_agg['time_window'] + 1
].dropna().copy()

print(f"{len(df_transitions)} valid consecutive transitions")
print("Next-state distribution:", df_transitions['State_next'].value_counts().to_dict())
df_transitions.head()

### Step 4 — Declare the DBN Structure

I declare the network using pgmpy's `DynamicBayesianNetwork`. Nodes are tuples
`(variable, time_slice)`, so `('State', 0)` is the current state and `('State', 1)`
is the next state. The current state and the three aggregate features point into the
next state. I also add self-transitions `(feature, 0) → (feature, 1)`, because
`DBN.fit` requires every variable to exist at both time slices.

In [ ]:
dbn = DBN()
edges  = [(('State', 0), ('State', 1))]
edges += [((c, 0), ('State', 1)) for c in AGG_CATS]   # aggregates -> next State
edges += [((c, 0), (c, 1))       for c in AGG_CATS]   # self-transitions
dbn.add_edges_from(edges)

print("DBN structure defined. Inter-slice edges into the next State:")
for (src, t0), (tgt, t1) in dbn.get_inter_edges():
    if tgt == 'State':
        print(f"  ({src}, {t0})  ->  ({tgt}, {t1})")

### Step 5 — Train the DBN with `dbn.fit()`

Because `State` now has only a few parents (the three aggregates plus the previous
state), its table is small and I can train the DBN **directly** with `dbn.fit()`
using Maximum Likelihood Estimation. I first put the data into the
`(variable, time_slice)` tuple format that pgmpy expects.

In [ ]:
data = pd.DataFrame()
data[('State', 0)] = df_transitions['State']
for c in AGG_CATS: data[(c, 0)] = df_transitions[c]
data[('State', 1)] = df_transitions['State_next']
for c in AGG_CATS: data[(c, 1)] = df_transitions[c]

dbn.fit(data, estimator='MLE')
print("dbn.fit() succeeded! CPTs learned. Model valid:", dbn.check_model())

## Inference & Evaluation

### Inference

I use `DBNInference` on the trained DBN. Given evidence (the current state and the
current aggregate features), it returns the probability distribution over
`('State', 1)`. `DBNInference` labels states as integers, so I read the index of
`Failure` from the trained CPT rather than from the returned factor.

### Evaluation

I evaluate on all transitions (in-sample), positive class = Failure, using accuracy,
precision, recall, F1 and the confusion matrix.

In [ ]:
infer    = DBNInference(dbn)
fail_idx = dbn.get_cpds(('State', 1)).state_names[('State', 1)].index('Failure')

def predict_failure(state_t, agg_values):
    """Returns P(State_t+1 = Failure | evidence)."""
    ev = {('State', 0): state_t}
    ev.update({(c, 0): v for c, v in agg_values.items()})
    try:
        q = infer.query(variables=[('State', 1)], evidence=ev)[('State', 1)]
        return float(q.values[fail_idx])
    except Exception:
        return 0.5

# Example query: a busy window with a long alarm
example = {'total_alarms_cat': 'High', 'total_dur_cat': 'High',
           'num_active_cat': 'High', 'max_dur_cat': 'High'}
print("P(Failure | State=Running, high alarm activity) =",
      round(predict_failure('Running', example), 3))

In [ ]:
tp = fp = fn = tn = 0
for _, r in df_transitions.iterrows():
    aggv = {c: r[c] for c in AGG_CATS}
    pred = 'Failure' if predict_failure(r['State'], aggv) >= 0.5 else 'Running'
    true = r['State_next']
    tp += (true == 'Failure' and pred == 'Failure')
    fp += (true == 'Running' and pred == 'Failure')
    fn += (true == 'Failure' and pred == 'Running')
    tn += (true == 'Running' and pred == 'Running')

n    = tp + fp + fn + tn
prec = tp / (tp + fp) if (tp + fp) else 0.0
rec  = tp / (tp + fn) if (tp + fn) else 0.0
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
print("=" * 40)
print("PURE DBN (aggregated, all 94 alarms)")
print("=" * 40)
print(f"Accuracy  : {(tp + tn) / n:.3f}")
print(f"Precision : {prec:.3f}")
print(f"Recall    : {rec:.3f}")
print(f"F1 Score  : {f1:.3f}")
print(f"Confusion : TP={tp}  FP={fp}  FN={fn}  TN={tn}")

## Notes

- This **is** a real `dbn.fit()` + `DBNInference` model — no flattening and no
  Naive-Bayes approximation — and it uses **all 94 alarms** through the aggregate
  summaries.
- Recall is low: collapsing 94 alarms into three totals throws away *which* alarm
  fired, so failure signals tied to specific alarms get averaged out. That is the
  price of keeping the pure joint DBN tractable.